## Linear Regression; Matrix formulations

#### Objectives

- Formulate the least squares regression problem compactly as a matrix equation
- Solve for the exact coefficients "manually" using the matrix formulation of the analytical solution



#### Remarks

- For illustration only, we will not use a training and test split. For ML we would always want to do this.  
- Least squares regression, aka ordinary least squares, is about the only ML model where we can actually write down a formula for the coefficients that minimize the loss function. The solution has a beautiful geometric interpretation as the projection of the target vector on the subspace spanned by the predictors. The handout notes show the underlying math. That said, as we have seen, we can also use gradient descent methods to minimize the loss function - even when we can obtain it via a formula. One reason is pedagogical-we want to illustrate that we get the same solution in either case. Also, understanding how gradient descent works in this simple linear model goes a long way to inform intuition on the more complex models. A third reason is computational. Analytical solutions to least squares and related problems typically involve matrix inverses that are computationally very expensive and numerically unstable. So a numerical solution is often preferred. 


#### `cars` data
Comes with R (actually called `mtcars`) but on Brightspace. Its real data. 

In [37]:
import pandas as pd
import numpy as np

cars = pd.read_csv('../data/cars.csv')
cars.head()

,Unnamed: 0,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


We model `mpg` based on `drat` which is a rear wheel axle "differential gear" ratio, the weight in tons, `wt`, and whether the car is manual or automatic, `am`. We pick these for no particular reason other than they are roughly on the same scale and not too large or small, which will allow us to easily interpret our coefficients and not deal with other potential numerical issues.

As we have seen, we can write the model as

$$
\mathbf{Y} = \mathbf{X}\mathbf{\beta} + \mathbf{\varepsilon}
$$

where 
- $\mathbf{Y}$ is $n \times 1$ vector of `mpg` values,
- $\mathbf{X}$ is $n \times p+1$ data matrix, where $p$ is the number of predictors, with a column of 1's as the first column, for the intercept. Here, $\mathbf{X}$ is $n \times 4$. 
-  $\mathbf{\beta}$ $p + 1 \times 1$ vector of coefficients, including the intercept $\beta_0$. 
- $\mathbf{\varepsilon}$ is a vector of normally distributed errors. 

We *usually* do not have to deal with the errors when we are searching for a best fit, though they do have something to say about how good the fit may be. 

Lets form our target and data matrix


In [38]:
ones = np.ones((cars.shape[0], 1))
X = cars[['drat','wt', 'am']]  ### note the double brackets
X = np.hstack([ones, X])  ## Alternatively, we could use np.concatenate([ones, X], axis=1)
y = cars['mpg']

## Analytical solution

We can use the "hat matrix" formula, Eq. (11) in the `matrix_least_squares_gradient` handout. For your convenience reproduced here:

$$ 
 \hat{\beta} =   ({\mathbf X}^T {\mathbf X} )^{-1}   {\mathbf X}^T {\mathbf {Y}} 
$$
to find the best fit coefficients. Note again that linear regression is about the only ML model for which we have an analytical solution. 

### Exercise: 

- Use the formula to find the best fit solution. (The video shows how to find matrix inverses with `linalg` in `numpy`.)
- Now use `sklearn` `LinearRegression` to fit the model. Do you obtain the same coefficients? Keep in mind that the `sklearn` model will have both `coefficient` and `intercept` attributes. The $\beta$ s you found above will have $\beta_0$ as the coefficient. How is `sklearn` solving for the coefficients? 

In [39]:
from sklearn.linear_model import LinearRegression
# Analytical solution
Bhat = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)
# Sklearn solution
model = LinearRegression()
model.fit(X, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [40]:
print("--- Analytical Coefficients ---")
print(f"Intercept (beta_0): {Bhat[0]:.6f}")
print(f"drat      (beta_1): {Bhat[1]:.6f}")
print(f"wt        (beta_2): {Bhat[2]:.6f}")
print(f"am        (beta_3): {Bhat[3]:.6f}\n")

print("--- Sklearn Coefficients ---")
print(f"Intercept         : {model.intercept_:.6f}")
print(f"drat, wt, am      : {model.coef_}")

--- Analytical Coefficients ---
Intercept (beta_0): 29.896957
drat      (beta_1): 1.787978
wt        (beta_2): -4.941886
am        (beta_3): -0.831072

--- Sklearn Coefficients ---
Intercept         : 29.896957
drat, wt, am      : [ 0.          1.78797792 -4.94188614 -0.83107242]


### Exercise: Interpreting the coefficients

Show that $\beta_j$ can be found as follows:
Regress $x_j$ onto all the all the other predictors, take the residual, and regress $Y$ onto that
residual - a simple linear regression - to obtain an intercept and slope. The slope coefficient
you obtain is $\beta_j$. You can pick one $x_j$,  How might we interpret this?

My hypothesis for our interpretation is that we can find xj with the most influence via coefficient and intercept

#### What I want to do for exercise 2:
- run regression on the threee parameters: 'wt', 'disp', and 'am'
- regression on 'wt' alone
- residual regression 
- compare slopes and intercept 

In [41]:

import numpy as np
from sklearn.linear_model import LinearRegression

# Load data
cars = pd.read_csv('../data/cars.csv')
y = cars['mpg']

# -------------------------------------------------------------
# 1. Model A: Just Weight (Simple Regression)
# -------------------------------------------------------------
model_just_wt = LinearRegression().fit(cars[['wt']], y)
coef_just_wt = model_just_wt.coef_[0]

# -------------------------------------------------------------
# 2. Model B: Full Model with all 3 parameters
# -------------------------------------------------------------
X_full = cars[['wt', 'drat', 'am']]
model_full = LinearRegression().fit(X_full, y)
coef_full_wt = model_full.coef_[0]  # wt is index 0 here

# -------------------------------------------------------------
# 3. Model C: Residual Model 
# -------------------------------------------------------------

x_j = cars['wt'].values
X_other = cars[['drat', 'am']]


model_xj = LinearRegression()
model_xj.fit(X_other, x_j)
xj_pred = model_xj.predict(X_other)

r_xj = x_j - xj_pred

model_residual = LinearRegression()
model_residual.fit(r_xj.reshape(-1, 1), y)

beta_j_isolated = model_residual.coef_[0]

print(f"Isolated Residual Regression Slope       : {beta_j_isolated:.6f}")
print(f"Isolated Residual Regression Intercept   : {model_residual.intercept_}")
# -------------------------------------------------------------
# Display Results
# -------------------------------------------------------------
print(f" Model with JUST Weight (Simple Reg)     : {coef_just_wt:.4f}")
print(f"Just Weight Intercept                    : {model_just_wt.intercept_}")

print(f" Model with ALL 3 Parameters (Full Reg)  : {coef_full_wt:.4f}")
print(f" All 3 parameters Intercept              : {model_full.intercept_}")


Isolated Residual Regression Slope       : -4.941886
Isolated Residual Regression Intercept   : 20.090625
 Model with JUST Weight (Simple Reg)     : -5.3445
Just Weight Intercept                    : 37.28512616734204
 Model with ALL 3 Parameters (Full Reg)  : -4.9419
 All 3 parameters Intercept              : 29.89695701659653


I will also perform a sliding function on xj to see how each xj performs 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression


cars = pd.read_csv('../data/cars.csv')
y = cars['mpg']
features = ['wt', 'drat', 'am']

for feature in features:

    x_j = cars[feature].values
    other_features = [f for f in features if f != feature]
    X_other = cars[other_features]


    model_xj = LinearRegression()
    model_xj.fit(X_other, x_j)
    xj_pred = model_xj.predict(X_other)

    r_xj = x_j - xj_pred

    model_residual = LinearRegression()
    model_residual.fit(r_xj.reshape(-1, 1), y)

    beta_j_isolated = model_residual.coef_[0]
    print(f"Intercept for {feature} : {model_residual.intercept_}")
    print(f"Slope for {feature}     : {beta_j_isolated}")

Intercept for wt : 20.090625
Slope for wt     : -4.941886144301468
Intercept for drat : 20.090625000000003
Slope for drat     : 1.7879779242785017
Intercept for am : 20.090625000000003
Slope for am     : -0.8310724237059456


Interesting observation overall.
I am observing that weight has the most impact in my particular data ste. I appreciate the practice with residual regression.
Another observation is that each piece of data is visibly independent in the equation when using residual regression.